In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

from core.logging import configure_logging
from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from registry import RegistryClient

from gl.models import GLSegments
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from break_analysis.tools import RegistryTools
from break_analysis import BreakAnalysisAgent
from break_analysis.models import BreakRecord
from break_analysis.builder import BreakCaseBuilder


configure_logging()

def display_df(df):
    display(df.toPandas())

In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('break-agent-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/17 11:27:54 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/09/17 11:27:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-80918095-3560-47f2-82fc-4a7240cca8b8;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 63ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#

In [4]:
registry_client = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)

gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry_client)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

registry_tools = RegistryTools(registry_client=registry_client)

In [5]:
llm = ChatOllama(
    model='qwen3:14b-q4_K_M',
    temperature=0
)

agent = BreakAnalysisAgent(
    llm=llm,
    registry_tools=registry_tools
)

In [6]:
workflow_run_id = 'd3282a32-5edc-4d85-b48c-e0ea38d9a4de'

recon_df = recon.get_results(workflow_run_id)
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

breaks = []
for row in breaks_df.collect():
    segments = GLSegments(
        entity_cd = row['ENTITY_CD'],
        branch_cd = row['BRANCH_CD'],
        dept_cd = row['DEPT_CD'],
        gl_account = row['GL_ACCOUNT'],
        sub_account = row['SUB_ACCOUNT'],
        affiliate_cd = row['AFFILIATE_CD'],
        product_cd = row['PRODUCT_CD'],
        book_cd = row['BOOK_CD'],
        source_cd = row['SOURCE_CD'],
    )
    breaks.append(
        BreakRecord(
            recon_result_id= row['RECON_RESULT_ID'],
            workflow_run_id = row['WORKFLOW_RUN_ID'],
            as_of_date = row['AS_OF_DATE'],
            segments = segments,
            accounted_currency = row['ACCOUNTED_CURRENCY'],
            interface_balance = row['INTERFACE_BALANCE'],
            gl_balance = row['GL_BALANCE'],
            difference_amount = row['DIFFERENCE_AMOUNT']
        )
    )

segment_defaults = gl.get_segment_defaults()
case_builder = BreakCaseBuilder(segment_defaults)

break_cases = case_builder.build(breaks)

2026-09-17 11:27:59,293 | INFO | break_analysis.builder | Building break cases | records=7
2026-09-17 11:27:59,294 | INFO | break_analysis.builder | Break cases built | total=3 | one_to_one=2 | many_to_one=1 | ambiguous=0 | interface_only=0 | gl_only=0 | unmatched=0 | duration_ms=1


In [7]:
break_case = break_cases[0]

agent.analyze(break_case)

2026-09-17 11:27:59,298 | INFO | break_analysis.agent | Analyzing break case | case_id=cbcb441d | workflow_run_id=d3282a32 | topology=MANY_TO_ONE | records=3
2026-09-17 11:27:59,298 | INFO | break_analysis.agent | Invoking LLM | case_id=cbcb441d | round=1
2026-09-17 11:28:14,662 | INFO | break_analysis.agent | LLM invoked | case_id=cbcb441d | round=1 | tool_calls=4 | input_tokens=1105 | output_tokens=203 | total_tokens=1308 | prompt_eval_count=1105 | eval_count=203 | load_ms=8018 | prompt_eval_ms=858 | eval_ms=6463 | total_ms=15347 | duration_ms=15359
2026-09-17 11:28:14,662 | INFO | break_analysis.agent | Tool round | case_id=cbcb441d | round=1 | tool_calls=4
2026-09-17 11:28:14,875 | INFO | break_analysis.agent | Tool invoked | case_id=cbcb441d | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "4100000", "business_dt": "2026-03-31"} | duration_ms=213
2026-09-17 11:28:15,004 | INFO | break_analysis.agent | Tool invoked | case_id=cbcb441d | tool=validate_se

BreakAnalysisResult(case_id=UUID('cbcb441d-d265-4bb4-9d88-e557222c605d'), recon_result_ids=(UUID('bf31b11b-2740-4649-8543-891f56e01821'), UUID('73c87bec-3637-4bda-b4bf-78af046119f6'), UUID('828d5971-42c9-40dc-b0e7-96e5faf562af')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation="The following segments in investigation records were invalid in Registry:\n1. GL_ACCOUNT '4100000' (missing in Registry)\n2. GL_SUB_ACCOUNT '004000' (inactive)\n3. GL_ACCOUNT '310000' (inactive)\n4. GL_SUB_ACCOUNT '003000' (inactive)")

In [8]:
break_case = break_cases[1]

agent.analyze(break_case)

2026-09-17 11:28:49,246 | INFO | break_analysis.agent | Analyzing break case | case_id=4e9d0893 | workflow_run_id=d3282a32 | topology=ONE_TO_ONE | records=2
2026-09-17 11:28:49,247 | INFO | break_analysis.agent | Invoking LLM | case_id=4e9d0893 | round=1
2026-09-17 11:28:51,561 | INFO | break_analysis.agent | LLM invoked | case_id=4e9d0893 | round=1 | tool_calls=1 | input_tokens=858 | output_tokens=50 | total_tokens=908 | prompt_eval_count=858 | eval_count=50 | load_ms=113 | prompt_eval_ms=294 | eval_ms=1684 | total_ms=2312 | duration_ms=2313
2026-09-17 11:28:51,561 | INFO | break_analysis.agent | Tool round | case_id=4e9d0893 | round=1 | tool_calls=1
2026-09-17 11:28:51,675 | INFO | break_analysis.agent | Tool invoked | case_id=4e9d0893 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "210000", "business_dt": "2026-03-31"} | duration_ms=114
2026-09-17 11:28:51,676 | INFO | break_analysis.agent | Invoking LLM | case_id=4e9d0893 | round=2
2026-09-17 11:28:5

BreakAnalysisResult(case_id=UUID('4e9d0893-869e-4590-a0c6-8d56e4de063d'), recon_result_ids=(UUID('8b947b0d-0d8b-4b7c-925c-2c0af4628151'), UUID('a8887fa1-74d8-46a2-878d-ebf5de37e8db')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation="The 'GL_ACCOUNT' segment with value '210000' exists in the Registry but is marked as inactive (status='I') for the business date 2026-03-31.")

In [9]:
break_case = break_cases[2]

agent.analyze(break_case)

2026-09-17 11:29:04,320 | INFO | break_analysis.agent | Analyzing break case | case_id=f88f6fe1 | workflow_run_id=d3282a32 | topology=ONE_TO_ONE | records=2
2026-09-17 11:29:04,321 | INFO | break_analysis.agent | Invoking LLM | case_id=f88f6fe1 | round=1
2026-09-17 11:29:09,398 | INFO | break_analysis.agent | LLM invoked | case_id=f88f6fe1 | round=1 | tool_calls=3 | input_tokens=860 | output_tokens=130 | total_tokens=990 | prompt_eval_count=860 | eval_count=130 | load_ms=115 | prompt_eval_ms=297 | eval_ms=4543 | total_ms=5076 | duration_ms=5077
2026-09-17 11:29:09,399 | INFO | break_analysis.agent | Tool round | case_id=f88f6fe1 | round=1 | tool_calls=3
2026-09-17 11:29:09,490 | INFO | break_analysis.agent | Tool invoked | case_id=f88f6fe1 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "", "business_dt": "2026-03-31"} | duration_ms=91
2026-09-17 11:29:09,578 | INFO | break_analysis.agent | Tool invoked | case_id=f88f6fe1 | tool=validate_segment | args={"

BreakAnalysisResult(case_id=UUID('f88f6fe1-0d38-410a-b0fe-50f3c46f9263'), recon_result_ids=(UUID('a0498239-ac79-4748-8ef6-6148cf9151b3'), UUID('4424b323-4067-433d-a23f-b534b6cf4a2e')), status=<BreakAnalysisStatus.UNEXPLAINED: 'UNEXPLAINED'>, root_cause=None, explanation='The hypothesis of REGISTRY_INVALID_SEGMENT is not supported because the investigation record contains blank values for gl_account, sub_account, and product_cd. Blank values are not classified as invalid Registry segments under the provided semantics.')

In [10]:
# spark.stop()